# 02 · Populations — `gwmock-pop`

Hand-writing CSVs does not scale. `gwmock-pop` samples astrophysical
source populations from a **directed-graph config**: every parameter is a
node with a *sampler* (a distribution) or a *transform* (derived from
other nodes via `@param` references). Sensible defaults ship for BBH,
BNS, and NSBH.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isaac-cf-wong/20260706-leuven-gw-workshop-gwmock/blob/main/materials/notebooks/02-populations.ipynb)

In [ ]:
# === Run this ONCE ===
# Installs gwmock on Google Colab / fresh kernels (needs Python 3.12-3.14).
# If gwmock is already available (e.g. you pre-installed it into a local
# venv), this skips the install and just reports the version.
# The [sgwb] extra pulls in the correlated-noise machinery used for the
# stochastic-background examples.
try:
    import gwmock
    print("gwmock already available:", gwmock.__version__)
except ModuleNotFoundError:
    %pip install -q "gwmock[sgwb]"

> **On Google Colab, the cell above may print red `pip` dependency-conflict
> warnings** about `gradio`, `numba`, or `huggingface-hub` wanting older
> `numpy` / `pydantic` / `typer`. Those are Colab's **preinstalled** packages,
> which this workshop never uses — **the gwmock install still succeeded, and
> you can ignore them.** (If a later cell ever errors with a `numpy`/ABI
> message, use *Runtime → Restart session* and re-run from the top.)

## 0. A word on precision

JAX defaults to **float32**. That is fine for masses and angles, but GPS
coalescence times are ~10⁹ s, where float32 spacing is **128 s** — every
`coa_time` in a catalogue would collapse onto a coarse grid. Since
v0.10.3, importing `gwmock_pop` therefore **enables 64-bit JAX floats
automatically** (set `GWMOCK_POP_DISABLE_X64=1` before import if you ever
need JAX's 32-bit default back):

In [ ]:
import gwmock_pop
import jax.numpy as jnp

print("gwmock-pop", gwmock_pop.__version__)
print("array dtype:", jnp.asarray(1.0).dtype)   # float64 — safe for GPS times

## 1. Sample a default BBH population

In [ ]:
from gwmock_pop import CBCSimulator

sim = CBCSimulator(seed=42)
pop = sim.simulate(2000)

print(f"{len(pop)} parameters per event:")
print(sorted(pop.keys()))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
axes[0].hist(np.asarray(pop["detector_frame_mass_1"]), bins=50)
axes[0].set_xlabel(r"$m_1$ [M$_\odot$]")
axes[1].hist(np.asarray(pop["distance"]), bins=50)
axes[1].set_xlabel("distance [Mpc]")
axes[2].hist(np.asarray(pop["spin_1z"]), bins=50)
axes[2].set_xlabel(r"$s_{1z}$")
for ax in axes:
    ax.set_ylabel("count")
plt.suptitle("default BBH population (2000 draws)")
plt.tight_layout()
plt.show()

## 2. Presets — a GWTC-4-calibrated population

The package ships full population models as presets. `gwtc4` is a
BBH model calibrated to the GWTC-4 catalogue (planck-tapered broken power
law + two peaks in mass, conditional mass-ratio pairing, Gaussian
χ_eff-derived spins, uniform-in-comoving-volume distances):

In [ ]:
from gwmock_pop import GraphSimulator, list_presets

print("presets:", list_presets())

gwtc4 = GraphSimulator.from_preset("gwtc4", seed=1).simulate(2000)

plt.figure(figsize=(7, 4))
bins = np.linspace(0, 120, 80)
plt.hist(np.asarray(pop["detector_frame_mass_1"]), bins=bins, alpha=0.5, label="default flat-ish")
plt.hist(np.asarray(gwtc4["detector_frame_mass_1"]), bins=bins, alpha=0.5, label="gwtc4 preset")
plt.xlabel(r"$m_1$ [M$_\odot$]")
plt.ylabel("count")
plt.legend()
plt.title("primary mass: default vs GWTC-4 preset")
plt.show()

## 3. Override any node — the graph model in action

Every parameter's prior can be swapped from Python (or YAML) without
touching the rest of the graph. A power-law + peak primary mass:

In [ ]:
sim_plp = CBCSimulator(seed=42, parameters={
    "detector_frame_mass_1": {"sampler": {
        "function": "power_law_plus_peak",
        "arguments": {"alpha": 3.5, "minimum": 5.0, "maximum": 100.0,
                      "lambda_peak": 0.1, "peak_mean": 35.0, "peak_sigma": 5.0,
                      "peak_maximum": 100.0}}},
})
pop_plp = sim_plp.simulate(5000)

plt.figure(figsize=(7, 4))
plt.hist(np.asarray(pop_plp["detector_frame_mass_1"]), bins=100)
plt.xlabel(r"$m_1$ [M$_\odot$]")
plt.ylabel("count")
plt.title("power-law + peak primary mass (see the 35 M$_\odot$ bump?)")
plt.show()

Transforms chain nodes together. Cosmologically meaningful distances:
sample redshift from a Madau–Dickinson star-formation-rate profile, then
*transform* it to luminosity distance:

In [ ]:
sim_z = CBCSimulator(seed=42, parameters={
    "redshift": {"sampler": {"function": "madau_dickinson_redshift",
                             "arguments": {"z_max": 3.0}}},
    "distance": {"transform": {"function": "redshift_to_luminosity_distance",
                               "arguments": {"redshift": "@redshift"}}},
})
pop_z = sim_z.simulate(5000)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(np.asarray(pop_z["redshift"]), bins=60)
axes[0].set_xlabel("redshift $z$")
axes[1].hist(np.asarray(pop_z["distance"]) / 1000, bins=60)
axes[1].set_xlabel("luminosity distance [Gpc]")
for ax in axes:
    ax.set_ylabel("count")
plt.suptitle("Madau–Dickinson redshifts → luminosity distances")
plt.tight_layout()
plt.show()

## 4. Other source types + the CLI

`BNSSimulator` / `NSBHSimulator` work the same way (BNS adds tidal
deformabilities `lambda_1`, `lambda_2`). And everything is scriptable from
the shell:

In [ ]:
from gwmock_pop import BNSSimulator

bns = BNSSimulator(seed=3).simulate(5)
print("BNS masses:", np.round(np.asarray(bns["detector_frame_mass_1"]), 2))
print("lambda_1  :", np.round(np.asarray(bns["lambda_1"]), 1))

In [ ]:
!gwmock-pop simulate --config gwtc4 --n 1000 --output pop_gwtc4.csv --seed 42 2>&1 | tail -1
!gwmock-pop inspect --config gwtc4 --n 1000 --seed 42 2>&1 | head -20

## 5. Write a catalogue gwmock can load

`write_population_catalogue` writes the CSV/HDF5 that
`FilePopulationLoader` (notebook 01) consumes — notebook 05 chains the
two.

In [ ]:
from gwmock_pop import read_population_catalogue, write_population_catalogue

write_population_catalogue("my_population.csv", pop_plp)
back = read_population_catalogue("my_population.csv")
print("round-trip OK:", len(next(iter(back.values()))), "events")

## ✏️ Try it (5 min)

1. Give the **secondary** mass a peak at 20 M☉ (override
   `detector_frame_mass_2` with `power_law_plus_peak`).
2. Sample the `nsbh_flat` preset (`GraphSimulator.from_preset`) and
   histogram both masses on the same axes.
3. See the float32 quantization for yourself: restart the kernel, set
   `os.environ["GWMOCK_POP_DISABLE_X64"] = "1"` **before** importing
   `gwmock_pop`, sample `coa_time` uniformly over a GPS-scale window, and
   print `np.unique(...)` — every event lands on the same 128 s grid
   point.

Next: **03 · Signals** — turn parameters into strain.